# Z3 (C# / .NET) 17 — Théorie des tableaux : Select, Store et axiomes de McCarthy

**Navigation** : [Index Z3-API](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Jumeau Python](Z3-17-Array-Theory-Python.ipynb) | [Série Z3 déclarative (Linq2Z3)](../Z3-Linq2Z3/README.md) | [→ 18 - Sudoku modes (Python)](Z3-18-Sudoku-Modes-Python.ipynb) | [<< 16e - Optimize](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb)
## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Modéliser un **tableau symbolique** avec la théorie des tableaux de Z3 (`ctx.MkArrayConst`, `ctx.MkSelect`, `ctx.MkStore`).
2. Énoncer et **vérifier les axiomes de McCarthy** (_read-over-write_) qui fondent la sémantique du `Store`.
3. Résoudre des contraintes sur les **contenus** d'un tableau (tableau trié, tableau égal à un autre, permutation) sans énumérer les valeurs.

Ce notebook est le jumeau C# (`Microsoft.Z3`) du notebook Python [`Z3-17-Array-Theory-Python`](Z3-17-Array-Theory-Python.ipynb) — il rejoue le même fil pédagogique via l'**API impérative** `Microsoft.Z3` (miroir direct de `pyz3`, et non la variante LINQ de la série [`Z3-Linq2Z3`](../Z3-Linq2Z3/README.md)). Il introduit une **théorie SMT distincte** des notebooks précédents (Booléens, arithmétique, chaînes) : la théorie des tableaux, où un tableau est vu comme une **fonction** Index → Élément.

> **Kernel** : `.net-csharp` (.NET Interactive). Le moteur est le **vrai** [Microsoft.Z3](https://github.com/Z3Prover/z3) via NuGet `#r "nuget: Microsoft.Z3"` (v4.12.2.0) — le même solveur C++ que `z3-solver` Python (dont le jumeau Python de cette série utilise la 4.16.0).


In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python)
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");
Console.WriteLine("Theorie des tableaux : ctx.MkArrayConst, ctx.MkSelect, ctx.MkStore.");
Console.WriteLine("Note de version : le NuGet Microsoft.Z3 est en 4.12.2.0, le jumeau Python utilise z3-solver 4.16.0.");

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Theorie des tableaux : ctx.MkArrayConst, ctx.MkSelect, ctx.MkStore.


Note de version : le NuGet Microsoft.Z3 est en 4.12.2.0, le jumeau Python utilise z3-solver 4.16.0.


## 1. Qu'est-ce que la théorie des tableaux ?

Un **tableau** en SMT n'est pas une structure de données mutable : c'est une **fonction totale** `Index → Élément`. Z3 expose cette théorie via le sort `Array(I, E)` et deux opérateurs canoniques :

| Opérateur | Signature | Sens |
|-----------|-----------|------|
| `ctx.MkSelect(A, i)` | `Array(I,E) × I → E` | lire l'élément d'indice `i` (`A[i]`) |
| `ctx.MkStore(A, i, v)` | `Array(I,E) × I × E → Array(I,E)` | renvoyer un **nouveau** tableau égal à `A` sauf en `i` où il vaut `v` |

Deux conséquences essentielles :

- **Immuable** : `Store` ne modifie pas `A`, il produit un autre tableau. C'est la sémantique *read-over-write* des tableaux fonctionnels.
- **Axiomes de McCarthy** : la théorie est définie par deux axiomes (vérifiés ci-dessous) qui lient `Select` et `Store`. Le solveur les utilise pour raisonner sur les contenus sans énumérer.

On modélise ainsi des structures indexées (tampons, mémoires, grilles) et on pose des contraintes sur leurs **contenus** — par exemple « le tableau est trié », « deux tableaux coïncident partout sauf en un indice » — que le solveur satisfait ou prouve impossibles.


## 2. Premier terrain — `Select` : contraindre les éléments d'un tableau symbolique

Construisons un tableau symbolique `a : Array(Int, Int)` — via `ctx.MkArrayConst("a", ctx.MkIntSort(), ctx.MkIntSort())` — et exigeons qu'il vérifie quatre contraintes ponctuelles : `a[0] = 1`, `a[1] = 2`, `a[2] > a[0]`, et `a[3] = a[1] + a[2]`. Z3 doit trouver un tableau (une fonction) satisfaisant tout cela.


In [2]:
// Exemple 1 : Select — contraindre les éléments d'un tableau symbolique.
var s = ctx.MkSolver();

// Un tableau symbolique d'entiers indexé par des entiers.
ArrayExpr a = ctx.MkArrayConst("a", ctx.MkIntSort(), ctx.MkIntSort());

// MkSelect renvoie un Expr générique : le contenu de ce tableau est connu comme entier.
ArithExpr SelA(int index) => (ArithExpr)ctx.MkSelect(a, ctx.MkInt(index));

// Contraintes ponctuelles sur les contenus via Select.
s.Assert(ctx.MkEq(SelA(0), ctx.MkInt(1)));
s.Assert(ctx.MkEq(SelA(1), ctx.MkInt(2)));
s.Assert(ctx.MkGt(SelA(2), SelA(0)));                                  // a[2] > a[0]
s.Assert(ctx.MkEq(SelA(3), ctx.MkAdd(SelA(1), SelA(2))));               // a[3] = a[1] + a[2]

Console.WriteLine($"Statut : {s.Check()}");
var m = s.Model;
long ValeurA(Expr terme) => ((IntNum)m.Evaluate(terme)).Int64;
Console.WriteLine($"a[0] = {ValeurA(SelA(0))}");
Console.WriteLine($"a[1] = {ValeurA(SelA(1))}");
Console.WriteLine($"a[2] = {ValeurA(SelA(2))}");
Console.WriteLine($"a[3] = {ValeurA(SelA(3))}");
Console.WriteLine($"Verification : a[2] > a[0] = {ValeurA(SelA(2)) > ValeurA(SelA(0))} ; a[3] == a[1] + a[2] = {ValeurA(SelA(3)) == ValeurA(SelA(1)) + ValeurA(SelA(2))}");

Statut : SATISFIABLE


a[0] = 1


a[1] = 2


a[2] = 40


a[3] = 42


Verification : a[2] > a[0] = True ; a[3] == a[1] + a[2] = True


### Lecture — le tableau comme fonction partielle

Z3 répond `SATISFIABLE` et produit un **modèle partiel** : il ne décrit `a` qu'aux indices qui apparaissent dans les contraintes (`0, 1, 2, 3`). Ailleurs, `a` est laissée libre (le modèle l'expose souvent comme une fonction par défaut, typiquement `K(Int, 0)` — le tableau constant nul). C'est exactement la puissance de la théorie : on ne déclare **pas** la taille du tableau, on ne déclare **pas** de valeurs par défaut ; on pose des contraintes sur les contenus, et le solveur synthétise une fonction cohérente.


## 3. Second terrain — les axiomes de McCarthy (`read-over-write`)

La sémantique du `Store` repose sur deux axiomes, dus à John McCarthy (1962) pour le λ-calcul avec tableaux. Soient `A` un tableau, `i, j` deux indices et `v` une valeur :

1. **Axiome d'égalité d'indice** : `ctx.MkSelect(ctx.MkStore(A, i, v), i) == v` — après écriture en `i`, la lecture en `i` renvoie la valeur écrite.
2. **Axiome de différence d'indice** : `i != j ⇒ ctx.MkSelect(ctx.MkStore(A, i, v), j) == ctx.MkSelect(A, j)` — écrire en `i` ne change aucun autre indice.

Ces deux axiomes sont **valides par construction** dans la théorie Z3 : on les vérifie en montrant que leur négation est `unsat`.


In [3]:
// Exemple 2 : vérifier les axiomes de McCarthy (négation unsat).
// Axiome 1 : Select(Store(A, i, v), i) == v.
var s1 = ctx.MkSolver();
var A1 = ctx.MkArrayConst("A1", ctx.MkIntSort(), ctx.MkIntSort());
var i1 = ctx.MkIntConst("i1");
var v1 = ctx.MkIntConst("v1");
var axiom1 = ctx.MkEq(ctx.MkSelect(ctx.MkStore(A1, i1, v1), i1), v1);
// Nier l'axiome : s'il existe un contre-exemple, la négation est sat.
s1.Assert(ctx.MkNot(axiom1));
Console.WriteLine($"Axiome 1 (negation doit etre UNSATISFIABLE) : {s1.Check()}");

// Axiome 2 : i != j  =>  Select(Store(A, i, v), j) == Select(A, j).
var s2 = ctx.MkSolver();
var A2 = ctx.MkArrayConst("A2", ctx.MkIntSort(), ctx.MkIntSort());
var i2 = ctx.MkIntConst("i2");
var j2 = ctx.MkIntConst("j2");
var v2 = ctx.MkIntConst("v2");
var axiom2 = ctx.MkImplies(
    ctx.MkNot(ctx.MkEq(i2, j2)),
    ctx.MkEq(ctx.MkSelect(ctx.MkStore(A2, i2, v2), j2), ctx.MkSelect(A2, j2)));
s2.Assert(ctx.MkNot(axiom2));
Console.WriteLine($"Axiome 2 (negation doit etre UNSATISFIABLE) : {s2.Check()}");

Axiome 1 (negation doit etre UNSATISFIABLE) : UNSATISFIABLE


Axiome 2 (negation doit etre UNSATISFIABLE) : UNSATISFIABLE


### Lecture — pourquoi ces axiomes sont `unsat` à la négation

Les deux négations sont `unsat` (`UNSATISFIABLE` dans l'API `Microsoft.Z3`) : **aucun** contre-exemple n'existe. Cela confirme que la théorie des tableaux de Z3 **incorpore** les axiomes de McCarthy — le solveur n'a pas besoin qu'on les ajoute explicitement, ils font partie de la définition du sort `Array`. C'est ce qui permet à Z3 de raisonner sur des suites de `Store` (une mémoire qui écrit plusieurs fois) de manière **déductive** plutôt qu'en énumérant les cas.


## 4. Troisième terrain — contraintes globales sur les contenus

Combinons `ctx.MkSelect`, `ctx.MkStore` et quantificateurs pour poser une contrainte **globale** : existe-t-il un tableau `t` de taille 5 (`t[0..4]`) strictement **croissant** ? On borne les indices par une contrainte de domaine, puis on exprime la croissance par un quantificateur borné.


In [4]:
// Exemple 3 : synthétiser un tableau croissant de taille 5.
var s3 = ctx.MkSolver();
ArrayExpr t = ctx.MkArrayConst("t", ctx.MkIntSort(), ctx.MkIntSort());

// MkSelect renvoie un Expr générique : le contenu de ce tableau est connu comme entier.
ArithExpr SelT(int index) => (ArithExpr)ctx.MkSelect(t, ctx.MkInt(index));

// Croissance stricte sur les indices consécutifs 0..3 (donc t[0] < t[1] < ... < t[4]).
for (int k = 0; k < 4; k++)
    s3.Assert(ctx.MkLt(SelT(k), SelT(k + 1)));

// On fixe un domaine raisonnable pour obtenir un modèle lisible.
for (int k = 0; k < 5; k++)
{
    s3.Assert(ctx.MkGe(SelT(k), ctx.MkInt(0)));
    s3.Assert(ctx.MkLe(SelT(k), ctx.MkInt(100)));
}

Console.WriteLine($"Statut : {s3.Check()}");
var m3 = s3.Model;

// Lecture du témoin synthétisé (valeur arbitraire parmi les modèles sat).
var valeurs = new long[5];
for (int k = 0; k < 5; k++)
    valeurs[k] = ((IntNum)m3.Evaluate(SelT(k), true)).Int64;
Console.WriteLine("Tableau croissant synthétisé : [" + string.Join(", ", valeurs) + "]");

// Démonstration Store : construire t2 = t où l'indice 2 est forcé à 42, et vérifier.
ArrayExpr t2 = ctx.MkStore(t, ctx.MkInt(2), ctx.MkInt(42));
ArithExpr SelT2(int index) => (ArithExpr)ctx.MkSelect(t2, ctx.MkInt(index));
long t2_2 = ((IntNum)m3.Evaluate(SelT2(2), true)).Int64;
long t2_0 = ((IntNum)m3.Evaluate(SelT2(0), true)).Int64;
Console.WriteLine($"Après Store(t, 2, 42) -> Select(t2, 2) = {t2_2}");
Console.WriteLine($"Select(t2, 0) inchangé = {t2_0} (== Select(t, 0) = {valeurs[0]})");

// Vérification par la cellule elle-même (témoin libre, mais contraintes respectées).
bool croissant = true;
for (int k = 0; k < 4; k++) if (!(valeurs[k] < valeurs[k + 1])) croissant = false;
bool storeOk = (t2_2 == 42) && (t2_0 == valeurs[0]);
Console.WriteLine($"Croissance stricte respectée : {croissant} ; axiomes du Store respectés (t2[2]=42 et t2[0]=t[0]) : {storeOk}");

Statut : SATISFIABLE


Tableau croissant synthétisé : [56, 57, 58, 59, 60]


Après Store(t, 2, 42) -> Select(t2, 2) = 42


Select(t2, 0) inchangé = 56 (== Select(t, 0) = 56)


Croissance stricte respectée : True ; axiomes du Store respectés (t2[2]=42 et t2[0]=t[0]) : True


### Lecture — `Store` produit un nouveau tableau cohérent, lu statut par statut

Le solveur synthétise un tableau croissant — les valeurs exactes sont celles affichées par la cellule précédente (`Tableau croissant synthetise : [...]`) : c'est un **témoin arbitraire**, Z3 est libre de choisir la base, et le jumeau Python a ainsi obtenu `[92, 93, 94, 95, 96]` sur son exécution. Ensuite, `ctx.MkStore(t, 2, 42)` crée `t2` : on lit `Select(t2, 2) == 42` (axiome 1) et `Select(t2, 0) == Select(t, 0)` (axiome 2, car `0 != 2`) — la cellule affiche les deux valeurs et vérifie explicitement ces deux propriétés. C'est la sémantique *read-over-write* en action : une suite d'écritures construit une **chaîne de tableaux immuables**, chacun dérivé du précédent, que le solveur sait inspecter sans ambiguïté.

Le modèle se lit aussi **statut par statut** — fixe, dérivé, libre. Au §2 (`a[0]=1`, `a[1]=2` en égalités, `a[2]>a[0]`, `a[3]=a[1]+a[2]`), le modèle rend 1, 2, une valeur **libre** pour `a[2]` (21240 sur le jumeau Python ; le solveur .NET peut en retenir une autre, la contrainte `a[2] > a[0]` est la seule qui la borne), et `a[3]` **dérivé** (l'arithmétique se referme : `a[3] = a[1] + a[2] = 2 + a[2]`). Ici au §4, même structure : la base du tableau croissant est **libre** (Z3 aurait pu choisir une autre base), `t2[2]=42` est **imposé** par le `Store`, `t2[0]` **inchangé** par l'axiome 2. Un modèle sat ne se lit donc pas valeur par valeur mais statut par statut : seule la part fixée et dérivée est une information ; la part libre changera d'une exécution à l'autre sans rien changer au verdict.


## Exercices

Les trois exercices vous font réutiliser `ctx.MkSelect`, `ctx.MkStore` et les axiomes de McCarthy sur de nouvelles contraintes. Chaque exercice suit le squelette : déclarer un tableau via `ctx.MkArrayConst`, ajouter des contraintes via `ctx.MkSelect`/`ctx.MkStore`, appeler `Check()` puis lire le modèle.

**Rappel C.1** : les stubs ne lèvent **jamais** d'erreur — `Console.WriteLine("Exercice à compléter")` et une valeur neutre (`return null`). Le notebook s'exécute de bout en bout même exercices non résolus.


### Exercice 1 — Deux tableaux égaux partout sauf en un indice

Étant donné deux tableaux `p` et `q` (`Array(Int, Int)`), ajoutez les contraintes pour que `p` et `q` coïncident en tous les indices `0..4` **sauf** en l'indice 2 où ils diffèrent. Vérifiez `SATISFIABLE` et affichez `p[2]` vs `q[2]` (ils doivent être différents) ainsi que `p[0]` vs `q[0]` (ils doivent être égaux).

**Indice** : `ctx.MkEq(ctx.MkSelect(p, k), ctx.MkSelect(q, k))` pour `k != 2`, et `ctx.MkNot(ctx.MkEq(ctx.MkSelect(p, 2), ctx.MkSelect(q, 2)))`.


In [5]:
// EXERCICE 1 : deux tableaux egaux partout sauf en l'indice 2.
string Exercice1(Context ctx)
{
    Console.WriteLine("Exercice 1 a completer : deux tableaux egaux sauf en l'indice 2.");
    // TODO etudiant :
    // 1. p = ctx.MkArrayConst("p", ctx.MkIntSort(), ctx.MkIntSort());
    //    q = ctx.MkArrayConst("q", ctx.MkIntSort(), ctx.MkIntSort());
    // 2. Pour k dans 0..4 sauf 2 : ctx.Assert(ctx.MkEq(ctx.MkSelect(p, ctx.MkInt(k)), ctx.MkSelect(q, ctx.MkInt(k))))
    // 3. Ajouter ctx.Assert(ctx.MkNot(ctx.MkEq(ctx.MkSelect(p, ctx.MkInt(2)), ctx.MkSelect(q, ctx.MkInt(2)))))
    // 4. s.Check(), lire p[2], q[2], p[0], q[0]
    return null;  // TODO etudiant : remplacer par la synthese (p[2], q[2], p[0], q[0])
}

var ex1 = Exercice1(ctx);
Console.WriteLine($"Exercice 1 : {(ex1 ?? "(a completer)")}");

Exercice 1 a completer : deux tableaux egaux sauf en l'indice 2.


Exercice 1 : (a completer)


### Exercice 2 — Prouver un invariant de `Store`

Montrez (par `unsat` de la négation) la propriété : si `B = Store(A, i, v)` et `j != i`, alors `Select(B, j) == Select(A, j)` pour **tout** `j` différent de `i`. Construire la négation et vérifier qu'elle est `unsat`.

**Indice** : c'est l'axiome 2 généralisé ; `B` est défini comme `ctx.MkStore(A, i, v)`, et on nie `ctx.MkImplies(ctx.MkNot(ctx.MkEq(j, i)), ctx.MkEq(ctx.MkSelect(B, j), ctx.MkSelect(A, j)))`.


In [6]:
// EXERCICE 2 : invariant de Store (axiome 2 generalise).
string Exercice2(Context ctx)
{
    Console.WriteLine("Exercice 2 a completer : prouver l'invariant Select(Store(A,i,v),j) == Select(A,j) pour j != i.");
    // TODO etudiant :
    // 1. A = ctx.MkArrayConst("A", ctx.MkIntSort(), ctx.MkIntSort());
    //    i = ctx.MkIntConst("i"); j = ctx.MkIntConst("j"); v = ctx.MkIntConst("v");
    // 2. B = ctx.MkStore(A, i, v)
    // 3. propriete = ctx.MkImplies(ctx.MkNot(ctx.MkEq(j, i)),
    //                          ctx.MkEq(ctx.MkSelect(B, j), ctx.MkSelect(A, j)))
    // 4. s.Assert(ctx.MkNot(propriete)); Console.WriteLine($"{s.Check()}")  -> UNSATISFIABLE attendu
    return null;  // TODO etudiant : remplacer par le verdict du solveur
}

var ex2 = Exercice2(ctx);
Console.WriteLine($"Exercice 2 : {(ex2 ?? "(a completer)")}");

Exercice 2 a completer : prouver l'invariant Select(Store(A,i,v),j) == Select(A,j) pour j != i.


Exercice 2 : (a completer)


### Exercice 3 — Aucun tableau ne satisfait (réponse `unsat`)

Énoncez une contrainte **impossible** sur un tableau `u : Array(Int, Int)` et vérifiez que le solveur répond `unsat`. Par exemple : `u` strictement croissant sur `0..3` **et** `u[0] > u[4]` (contradictoire). Affichez le verdict.

**Question subsidiaire** : pourquoi `unsat` ici alors que l'exemple 3 était `sat` ?


In [7]:
// EXERCICE 3 : contrainte impossible -> reponse unsat.
string Exercice3(Context ctx)
{
    Console.WriteLine("Exercice 3 a completer : tableau croissant ET u[0] > u[4] -> attendu UNSATISFIABLE.");
    // TODO etudiant :
    // 1. u = ctx.MkArrayConst("u", ctx.MkIntSort(), ctx.MkIntSort())
    // 2. Croissance : ctx.Assert(ctx.MkLt(ctx.MkSelect(u, ctx.MkInt(k)), ctx.MkSelect(u, ctx.MkInt(k + 1)))) pour k dans 0..3
    // 3. Contradiction : ctx.Assert(ctx.MkGt(ctx.MkSelect(u, ctx.MkInt(0)), ctx.MkSelect(u, ctx.MkInt(4))))
    // 4. s.Check()  -> UNSATISFIABLE attendu
    return null;  // TODO etudiant : remplacer par le verdict du solveur
}

var ex3 = Exercice3(ctx);
Console.WriteLine($"Exercice 3 : {(ex3 ?? "(a completer)")}");

Exercice 3 a completer : tableau croissant ET u[0] > u[4] -> attendu UNSATISFIABLE.


Exercice 3 : (a completer)


## Conclusion

Ce notebook introduit la **théorie des tableaux** de Z3 — une théorie SMT distincte des arithmétiques et des chaînes, où un tableau est une fonction `Index → Élément` manipulée via `Select` et `Store`.

| Concept | API Z3 | Rôle |
|---------|--------|------|
| Tableau symbolique | `ctx.MkArrayConst("a", ctx.MkIntSort(), ctx.MkIntSort())` | fonction immuable Index → Élément |
| Lecture | `ctx.MkSelect(a, i)` | `a[i]` |
| Écriture | `ctx.MkStore(a, i, v)` | nouveau tableau égal à `a` sauf en `i` |
| Axiome 1 (McCarthy) | `ctx.MkSelect(ctx.MkStore(A,i,v),i) == v` | lu = écrit au même indice |
| Axiome 2 (McCarthy) | `i != j ⇒ ctx.MkSelect(ctx.MkStore(A,i,v),j) == ctx.MkSelect(A,j)` | écrit en `i` inchangé ailleurs |

**Quand la théorie devient discriminante** : sur des petites instances, `Select`/`Store` équivalent à de l'énumération. La théorie des tableaux prend tout son sens pour raisonner sur des **contenus sous contraintes globales** (invariants de tampon, égalités partielles, preuves d'impossibilité `unsat`) — exactement le terrain où le solveur SMT distingue un moteur de reconnaissance d'un moteur de **raisonnement déductif**.

### Voir aussi

- Jumeau Python de ce carnet : [`Z3-17-Array-Theory-Python.ipynb`](Z3-17-Array-Theory-Python.ipynb) — même fil pédagogique, API `pyz3`.
- Série [`Z3-Linq2Z3`](../Z3-Linq2Z3/README.md) : la variante **déclarative** (LINQ) de la théorie des tableaux — [`04_Array_Theory.ipynb`](../Z3-Linq2Z3/04_Array_Theory.ipynb) ; ce carnet reste le miroir direct de l'**API impérative** `Microsoft.Z3`, conformément à la convention de la série Z3-API.
- Notebooks voisins de la même série : [`Z3-03-Tactics-CSharp.ipynb`](Z3-03-Tactics-CSharp.ipynb) (théories BitVec et Array), [`Z3-15-Nested-Arrays-2D-Python.ipynb`](Z3-15-Nested-Arrays-2D-Python.ipynb) (tableaux imbriqués et grilles 2D).

***

*Jumeau C# créé pour l'issue jsboige/CoursIA#27102 (parent #26070, voir #26534). Source Python : `MyIA.AI.Notebooks/SymbolicAI/SMT/Z3-API/Z3-17-Array-Theory-Python.ipynb`. Noyau d'exécution : `.net-csharp`, moteur `Microsoft.Z3` 4.12.2.0 (NuGet) via `#r "nuget: Microsoft.Z3"` et `#load "Z3NativeLoader.cs"`. Les verdicts (sat / unsat) et les propriétés vérifiables des modèles sont identiques au jumeau Python ; les **témoins arbitraires** (valeur libre de `a[2]`, base du tableau croissant) peuvent différer, la version du moteur Python étant 4.16.0 contre 4.12.2.0 pour le NuGet.*
